# MedGS interpolation experiment

This notebook uses cases prepared by
`01_inspect_4d_lung_and_prepare_split_v1.ipynb`.

The preparation notebook must already have saved, for each case:

```text
results/prepared_cases/<config_name>/<case_name>/
├── case.json
└── split_manifest.csv
```

This notebook:

1. lists prepared configurations and cases;
2. loads one case explicitly by name;
3. converts only the training CT slices to the MedGS `original/` and
   `mirror/` PNG layout;
4. trains the image reconstruction pipeline;
5. renders one midpoint between consecutive training frames;
6. evaluates reconstructed test slices using PSNR, SSIM, and,
   when the `lpips` package is available, LPIPS;
7. compares MedGS with ordinary linear interpolation;
8. provides an interactive sequence:
   **train input → reconstructed test slice → train input → ...**

Test DICOM slices are never copied into the MedGS training input.
They are read only after rendering, for evaluation.

Run the notebook on an allocated GH200 worker with the MedGS kernel.
To reopen an already completed run without retraining, set
`RUN_TRAINING = False` and `RUN_RENDERING = False`.

In [2]:
import sys

print(sys.executable)

/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/envs/medgs-gh200/bin/python


In [18]:
print("Installing LPIPS in the active Jupyter kernel.")

%pip install lpips

Installing LPIPS in the active Jupyter kernel.
Note: you may need to restart the kernel to use updated packages.


In [19]:
from __future__ import annotations

print("Importing libraries used by the MedGS experiment.")

import json
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path
import lpips
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pydicom
import torch
from IPython.display import display
from PIL import Image
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 500)

Importing libraries used by the MedGS experiment.


## 1. Project paths and runtime

In [3]:
print("Defining project, prepared-case, and experiment paths.")

STORAGE_ROOT = Path("/net/storage/pr3/plgrid/plggtriplane")
PROJECT_OWNER_LOGIN = "plgmozo"

SHARED_PROJECT_ROOT = STORAGE_ROOT / PROJECT_OWNER_LOGIN / "medgs4d"
USER_PROJECT_ROOT = STORAGE_ROOT / os.environ["USER"] / "medgs4d"

SERIES_ZIPS_ROOT = SHARED_PROJECT_ROOT / "data" / "raw" / "series_zips"
DICOM_ROOT = SHARED_PROJECT_ROOT / "data" / "raw" / "dicom_by_series"
METADATA_ROOT = SHARED_PROJECT_ROOT / "data" / "metadata"
RESULTS_ROOT = USER_PROJECT_ROOT / "results"
LOGS_ROOT = USER_PROJECT_ROOT / "logs"
REPOSITORIES_ROOT = SHARED_PROJECT_ROOT / "repo"
ENVIRONMENTS_ROOT = SHARED_PROJECT_ROOT / "envs"
SHARED_RESULTS_ROOT = SHARED_PROJECT_ROOT / "results"
SHARED_LOGS_ROOT = SHARED_PROJECT_ROOT / "logs"
CASES_ROOT = RESULTS_ROOT / "prepared_cases"

MEDGS_REPOSITORY = REPOSITORIES_ROOT / "MedGS"
EXPERIMENTS_ROOT = RESULTS_ROOT / "experiments"

path_table = pd.DataFrame(
    [
        {"Purpose": "Prepared cases", "Path": CASES_ROOT},
        {"Purpose": "DICOM input", "Path": DICOM_ROOT},
        {"Purpose": "MedGS repository", "Path": MEDGS_REPOSITORY},
        {"Purpose": "Experiment outputs", "Path": EXPERIMENTS_ROOT},
    ]
)

display(path_table)

Defining project, prepared-case, and experiment paths.


,Purpose,Path
0,Prepared cases,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/prepared_cases
1,DICOM input,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series
2,MedGS repository,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/repo/MedGS
3,Experiment outputs,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments


In [4]:
print("Checking the active Python kernel and GPU.")

runtime_df = pd.DataFrame(
    [
        {"Item": "Python executable", "Value": sys.executable},
        {"Item": "Slurm job", "Value": os.environ.get("SLURM_JOB_ID", "")},
        {"Item": "CUDA available", "Value": torch.cuda.is_available()},
        {"Item": "GPU", "Value": torch.cuda.get_device_name(0)},
        {"Item": "PyTorch", "Value": torch.__version__},
    ]
)

display(runtime_df)

Checking the active Python kernel and GPU.


,Item,Value
0,Python executable,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/envs/medgs-gh200/bin/python
1,Slurm job,19654179
2,CUDA available,True
3,GPU,NVIDIA GH200 120GB
4,PyTorch,2.7.1+cu128


## 2. List prepared experiment configurations

In [5]:
print("Listing all prepared experiment configurations.")

configuration_rows = []

for config_directory in sorted(
        path for path in CASES_ROOT.iterdir() if path.is_dir()
):
    case_files = sorted(config_directory.glob("*/case.json"))

    if not case_files:
        continue

    cases = [
        json.loads(case_file.read_text(encoding="utf-8"))
        for case_file in case_files
    ]

    patients = {case["patient_id"] for case in cases}
    phases = sorted({float(case["phase_percent"]) for case in cases})

    configuration_rows.append(
        {
            "ConfigName": config_directory.name,
            "PreparedCases": len(cases),
            "Patients": len(patients),
            "Phases": ", ".join(f"{phase:g}%" for phase in phases),
            "TotalTrainSlices": sum(
                int(case["train_slice_count"]) for case in cases
            ),
            "TotalTestSlices": sum(
                int(case["test_slice_count"]) for case in cases
            ),
            "Directory": str(config_directory),
        }
    )

prepared_configurations_df = (
    pd.DataFrame(configuration_rows)
    .sort_values("ConfigName")
    .reset_index(drop=True)
)

print(
    f"Prepared experiment configurations: "
    f"{len(prepared_configurations_df)}"
)
display(prepared_configurations_df)

Listing all prepared experiment configurations.
Prepared experiment configurations: 1


,ConfigName,PreparedCases,Patients,Phases,TotalTrainSlices,TotalTestSlices,Directory
0,single_phase_ct_every_second_slice_v1,1,1,20%,26,24,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/prepared_cases/single_phase_ct_every_second_slice_v1


## 3. Select a configuration and list its cases

Set `CONFIG_NAME` to one of the values displayed above.

In [6]:
print("Listing prepared cases for the selected configuration.")

CONFIG_NAME = "single_phase_ct_every_second_slice_v1"

config_directory = CASES_ROOT / CONFIG_NAME
case_rows = []

for case_config_path in sorted(config_directory.glob("*/case.json")):
    case_config = json.loads(
        case_config_path.read_text(encoding="utf-8")
    )

    case_rows.append(
        {
            "CaseName": case_config["resolved_case_name"],
            "PatientID": case_config["patient_id"],
            "StudyDate": case_config["study_date"],
            "PhasePercent": case_config["phase_percent"],
            "Slices": case_config["slice_count"],
            "Train": case_config["train_slice_count"],
            "Test": case_config["test_slice_count"],
            "SeriesDescription": case_config["series_description"],
            "CaseDirectory": str(case_config_path.parent),
        }
    )

prepared_cases_df = (
    pd.DataFrame(case_rows)
    .sort_values(["PatientID", "StudyDate", "CaseName"])
    .reset_index(drop=True)
)

print(
    f"Prepared cases for configuration {CONFIG_NAME}: "
    f"{len(prepared_cases_df)}"
)
display(prepared_cases_df)

Listing prepared cases for the selected configuration.
Prepared cases for configuration single_phase_ct_every_second_slice_v1: 1


,CaseName,PatientID,StudyDate,PhasePercent,Slices,Train,Test,SeriesDescription,CaseDirectory
0,single_phase_ct_every_second_slice_v1__112_HM10395__phase_20__series_036286623964,112_HM10395,19991207,20.0,50,26,24,"P4^P112^S121^I0, Gated, 20.0%",/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/prepared_cases/single_phase_ct_every_second_slice_v1/si...


## 4. Load one prepared case

Set `SELECTED_CASE_NAME` explicitly using a name from the table above.

In [7]:
print("Loading the selected prepared case for training and evaluation.")

SELECTED_CASE_NAME = (
    "single_phase_ct_every_second_slice_v1"
    "__112_HM10395"
    "__phase_20"
    "__series_036286623964"
)

selected_case_directory = (
        CASES_ROOT / CONFIG_NAME / SELECTED_CASE_NAME
)

loaded_case_config = json.loads(
    (selected_case_directory / "case.json").read_text(
        encoding="utf-8"
    )
)

split_manifest_df = pd.read_csv(
    selected_case_directory
    / loaded_case_config["manifest_file"]
)

split_manifest_df["Path"] = split_manifest_df[
    "DICOMRelativePath"
].map(lambda value: str(DICOM_ROOT / value))

split_manifest_df = split_manifest_df.sort_values(
    "SliceIndex"
).reset_index(drop=True)

train_manifest_df = (
    split_manifest_df.loc[
        split_manifest_df["Split"] == "train"
        ]
    .sort_values("SliceIndex")
    .reset_index(drop=True)
)

test_manifest_df = (
    split_manifest_df.loc[
        split_manifest_df["Split"] == "test"
        ]
    .sort_values("SliceIndex")
    .reset_index(drop=True)
)

PATIENT_ID = loaded_case_config["patient_id"]
TARGET_PHASE_PERCENT = float(
    loaded_case_config["phase_percent"]
)
HU_WINDOW_LOW = float(loaded_case_config["hu_window_low"])
HU_WINDOW_HIGH = float(loaded_case_config["hu_window_high"])
selected_study_uid = loaded_case_config["study_instance_uid"]
selected_series_uid = loaded_case_config["series_instance_uid"]
selected_series_dir = (
        DICOM_ROOT / loaded_case_config["series_relative_path"]
)
resolved_case_name = loaded_case_config["resolved_case_name"]

print(f"Case name:         {resolved_case_name}")
print(f"Patient ID:        {PATIENT_ID}")
print(f"Study date:        {loaded_case_config['study_date']}")
print(f"Study UID:         {selected_study_uid}")
print(f"Series UID:        {selected_series_uid}")
print(f"Respiratory phase: {TARGET_PHASE_PERCENT:g}%")
print(
    f"HU window:         "
    f"[{HU_WINDOW_LOW:g}, {HU_WINDOW_HIGH:g}]"
)
print(f"Train slices:      {len(train_manifest_df)}")
print(f"Test slices:       {len(test_manifest_df)}")
print(f"Series directory:  {selected_series_dir}")

display(
    split_manifest_df[
        [
            "SliceIndex",
            "Split",
            "SliceCoordinate",
            "LowerTrainCoordinate",
            "UpperTrainCoordinate",
            "File",
        ]
    ]
)

Loading the selected prepared case for training and evaluation.
Case name:         single_phase_ct_every_second_slice_v1__112_HM10395__phase_20__series_036286623964
Patient ID:        112_HM10395
Study date:        19991207
Study UID:         1.3.6.1.4.1.14519.5.2.1.6834.5010.204802741624618752298023624863
Series UID:        1.3.6.1.4.1.14519.5.2.1.6834.5010.569896177987482348036286623964
Respiratory phase: 20%
HU window:         [-1000, 400]
Train slices:      26
Test slices:       24
Series directory:  /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5010.569896177987482348036286623964


,SliceIndex,Split,SliceCoordinate,LowerTrainCoordinate,UpperTrainCoordinate,File
0,0,train,-73.5,-73.5,-73.5,00000008.dcm
1,1,test,-70.5,-73.5,-67.5,00000005.dcm
2,2,train,-67.5,-67.5,-67.5,00000022.dcm
3,3,test,-64.5,-67.5,-61.5,00000044.dcm
4,4,train,-61.5,-61.5,-61.5,00000003.dcm
5,5,test,-58.5,-61.5,-55.5,00000016.dcm
6,6,train,-55.5,-55.5,-55.5,00000042.dcm
7,7,test,-52.5,-55.5,-49.5,00000006.dcm
8,8,train,-49.5,-49.5,-49.5,00000025.dcm
9,9,test,-46.5,-49.5,-43.5,00000021.dcm


## 5. Experiment settings

The default run uses 30,000 training iterations. The current MedGS
training code starts its in-between-frame interpolation stage after
10,000 iterations, so a shorter smoke test does not exercise the full
interpolation training procedure.

Change `RUN_NAME` whenever training a new variant. Existing model
output is not overwritten automatically.

In [8]:
print("Defining MedGS training and rendering settings.")

ITERATIONS = 30_000
POLY_DEGREE = 2
BATCH_SIZE = 3
CAMERA = "mirror"
RENDER_INTERPOLATION = 2

RUN_TRAINING = True
RUN_RENDERING = True

RUN_NAME = f"medgs_img_poly{POLY_DEGREE}_{ITERATIONS}_v1"

RUN_ROOT = (
        EXPERIMENTS_ROOT
        / CONFIG_NAME
        / SELECTED_CASE_NAME
        / RUN_NAME
)
MEDGS_DATASET_ROOT = RUN_ROOT / "dataset"
ORIGINAL_IMAGES_ROOT = MEDGS_DATASET_ROOT / "original"
MIRROR_IMAGES_ROOT = MEDGS_DATASET_ROOT / "mirror"
MODEL_ROOT = RUN_ROOT / "model"
RENDER_ROOT = MODEL_ROOT / "render_img"

TRAIN_SCRIPT = MEDGS_REPOSITORY / "train.py"
RENDER_SCRIPT = MEDGS_REPOSITORY / "render.py"

METRICS_PATH = RUN_ROOT / "metrics.csv"
RUN_CONFIG_PATH = RUN_ROOT / "run.json"

RUN_ROOT.mkdir(parents=True, exist_ok=True)

settings_df = pd.DataFrame(
    [
        {"Setting": "Run name", "Value": RUN_NAME},
        {"Setting": "Iterations", "Value": ITERATIONS},
        {"Setting": "Polynomial degree", "Value": POLY_DEGREE},
        {"Setting": "Batch size", "Value": BATCH_SIZE},
        {"Setting": "Camera", "Value": CAMERA},
        {
            "Setting": "Render interpolation",
            "Value": RENDER_INTERPOLATION,
        },
        {"Setting": "Dataset", "Value": MEDGS_DATASET_ROOT},
        {"Setting": "Model output", "Value": MODEL_ROOT},
    ]
)

display(settings_df)

Defining MedGS training and rendering settings.


,Setting,Value
0,Run name,medgs_img_poly2_30000_v1
1,Iterations,30000
2,Polynomial degree,2
3,Batch size,3
4,Camera,mirror
5,Render interpolation,2
6,Dataset,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
7,Model output,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...


## 6. Convert training DICOM slices to MedGS input

In [9]:
print("Converting only training CT slices to HU-windowed MedGS PNG frames.")


def read_hu_image(path: Path) -> np.ndarray:
    """Read one CT slice and convert stored values to Hounsfield units."""

    dataset = pydicom.dcmread(str(path), force=True)
    pixels = dataset.pixel_array.astype(np.float32)
    slope = float(getattr(dataset, "RescaleSlope", 1.0))
    intercept = float(getattr(dataset, "RescaleIntercept", 0.0))
    return pixels * slope + intercept


def window_hu(
        image_hu: np.ndarray,
        low: float,
        high: float,
) -> np.ndarray:
    """Clip an HU image to one window and normalize it to [0, 1]."""

    clipped = np.clip(image_hu, low, high)
    return (clipped - low) / (high - low)


def grayscale_to_rgb_uint8(image: np.ndarray) -> np.ndarray:
    """Convert a normalized grayscale image to three-channel uint8 RGB."""

    grayscale = np.round(image * 255.0).astype(np.uint8)
    return np.repeat(grayscale[..., None], 3, axis=2)


shutil.rmtree(MEDGS_DATASET_ROOT, ignore_errors=True)
ORIGINAL_IMAGES_ROOT.mkdir(parents=True)
MIRROR_IMAGES_ROOT.mkdir(parents=True)

frame_rows = []

for frame_index, row in train_manifest_df.iterrows():
    normalized = window_hu(
        read_hu_image(Path(row["Path"])),
        HU_WINDOW_LOW,
        HU_WINDOW_HIGH,
    )
    rgb = grayscale_to_rgb_uint8(normalized)

    filename = f"{frame_index:04d}.png"
    original_path = ORIGINAL_IMAGES_ROOT / filename
    mirror_path = MIRROR_IMAGES_ROOT / filename

    Image.fromarray(rgb).save(original_path)
    Image.fromarray(np.fliplr(rgb)).save(mirror_path)

    frame_rows.append(
        {
            "FrameIndex": frame_index,
            "SliceIndex": int(row["SliceIndex"]),
            "SliceCoordinate": float(row["SliceCoordinate"]),
            "DICOMRelativePath": row["DICOMRelativePath"],
            "OriginalPNG": str(original_path),
            "MirrorPNG": str(mirror_path),
        }
    )

frame_manifest_df = pd.DataFrame(frame_rows)

print(f"Training frames written: {len(frame_manifest_df)}")
print(f"Original directory:      {ORIGINAL_IMAGES_ROOT}")
print(f"Mirror directory:        {MIRROR_IMAGES_ROOT}")

display(
    frame_manifest_df[
        [
            "FrameIndex",
            "SliceIndex",
            "SliceCoordinate",
            "DICOMRelativePath",
        ]
    ]
)

Converting only training CT slices to HU-windowed MedGS PNG frames.
Training frames written: 26
Original directory:      /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/single_phase_ct_every_second_slice_v1__112_HM10395__phase_20__series_036286623964/medgs_img_poly2_30000_v1/dataset/original
Mirror directory:        /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/single_phase_ct_every_second_slice_v1__112_HM10395__phase_20__series_036286623964/medgs_img_poly2_30000_v1/dataset/mirror


,FrameIndex,SliceIndex,SliceCoordinate,DICOMRelativePath
0,0,0,-73.5,112_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5010.569896177987482348036286623964/00000008.dcm
1,1,2,-67.5,112_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5010.569896177987482348036286623964/00000022.dcm
2,2,4,-61.5,112_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5010.569896177987482348036286623964/00000003.dcm
3,3,6,-55.5,112_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5010.569896177987482348036286623964/00000042.dcm
4,4,8,-49.5,112_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5010.569896177987482348036286623964/00000025.dcm
5,5,10,-43.5,112_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5010.569896177987482348036286623964/00000015.dcm
6,6,12,-37.5,112_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5010.569896177987482348036286623964/00000050.dcm
7,7,14,-31.5,112_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5010.569896177987482348036286623964/00000012.dcm
8,8,16,-25.5,112_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5010.569896177987482348036286623964/00000023.dcm
9,9,18,-19.5,112_HM10395/1.3.6.1.4.1.14519.5.2.1.6834.5010.569896177987482348036286623964/00000041.dcm


## 7. Train MedGS

In [10]:
print("Running MedGS image reconstruction training.")

# Train the MedGS model for 30,000 updates on three images at a time, modeling the change between slices with a quadratic polynomial using the original and mirror images, and then generate one intermediate slice between each pair of training slices.

train_command = [
    sys.executable,
    str(TRAIN_SCRIPT),
    "-s",
    str(MEDGS_DATASET_ROOT),
    "-m",
    str(MODEL_ROOT),
    "--pipeline",
    "img",
    "--iterations",
    str(ITERATIONS),
    "--poly_degree",
    str(POLY_DEGREE),
    "--batch_size",
    str(BATCH_SIZE),
    "--camera",
    CAMERA,
    "--test_iterations",
    str(ITERATIONS),
    "--save_iterations",
    str(ITERATIONS),
    "--checkpoint_iterations",
    str(ITERATIONS),
]

print("Command:")
print(" ".join(shlex.quote(part) for part in train_command))

if RUN_TRAINING:
    if MODEL_ROOT.exists():
        raise FileExistsError(
            f"Model output already exists: {MODEL_ROOT}\n"
            "Change RUN_NAME before starting a new training run."
        )

    subprocess.run(
        train_command,
        cwd=MEDGS_REPOSITORY,
        check=True,
    )
else:
    print("Training skipped; the existing model output will be used.")

Running MedGS image reconstruction training.
Command:
/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/envs/medgs-gh200/bin/python /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/repo/MedGS/train.py -s /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/single_phase_ct_every_second_slice_v1__112_HM10395__phase_20__series_036286623964/medgs_img_poly2_30000_v1/dataset -m /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/single_phase_ct_every_second_slice_v1__112_HM10395__phase_20__series_036286623964/medgs_img_poly2_30000_v1/model --pipeline img --iterations 30000 --poly_degree 2 --batch_size 3 --camera mirror --test_iterations 30000 --save_iterations 30000 --checkpoint_iterations 30000
torch cuda:  True
Optimizing /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/single_phase_ct_every_second_slice_v1

Training progress: 100%|██████████| 30000/30000 [22:20<00:00, 22.37it/s, Loss=0.0189703, psnr=42.27, point=1341829]


prev_next_overlap 2 [19/07 19:22:51]

[ITER 600] Densifying Gaussians [19/07 19:23:05]

[ITER 700] Densifying Gaussians [19/07 19:23:07]

[ITER 800] Densifying Gaussians [19/07 19:23:10]

[ITER 900] Densifying Gaussians [19/07 19:23:12]

[ITER 1000] Densifying Gaussians [19/07 19:23:14]

[ITER 1100] Densifying Gaussians [19/07 19:23:16]

[ITER 1200] Densifying Gaussians [19/07 19:23:18]

[ITER 1300] Densifying Gaussians [19/07 19:23:21]

[ITER 1400] Densifying Gaussians [19/07 19:23:23]

[ITER 1500] Densifying Gaussians [19/07 19:23:26]

[ITER 1600] Densifying Gaussians [19/07 19:23:28]

[ITER 1700] Densifying Gaussians [19/07 19:23:31]

[ITER 1800] Densifying Gaussians [19/07 19:23:34]

[ITER 1900] Densifying Gaussians [19/07 19:23:37]

[ITER 2000] Densifying Gaussians [19/07 19:23:40]

[ITER 2100] Densifying Gaussians [19/07 19:23:43]

[ITER 2200] Densifying Gaussians [19/07 19:23:46]

[ITER 2300] Densifying Gaussians [19/07 19:23:49]

[ITER 2400] Densifying Gaussians [19/07 19:23:52

## 8. Render training frames and midpoint reconstructions

In [11]:
print("Rendering trained frames and one midpoint between adjacent frames.")

render_command = [
    sys.executable,
    str(RENDER_SCRIPT),
    "--model_path",
    str(MODEL_ROOT),
    "--iteration",
    "-1",
    "--interp",
    str(RENDER_INTERPOLATION),
    "--pipeline",
    "img",
    "--camera",
    CAMERA,
    "--poly_degree",
    str(POLY_DEGREE),
    "--chunks",
    "1",
]

print("Command:")
print(" ".join(shlex.quote(part) for part in render_command))

if RUN_RENDERING:
    shutil.rmtree(RENDER_ROOT, ignore_errors=True)

    subprocess.run(
        render_command,
        cwd=MEDGS_REPOSITORY,
        check=True,
    )
else:
    print("Rendering skipped; the existing render directory will be used.")

rendered_files = sorted(RENDER_ROOT.glob("*.png"))

print(f"Rendered PNG files: {len(rendered_files)}")
print(f"Render directory:   {RENDER_ROOT}")

Rendering trained frames and one midpoint between adjacent frames.
Command:
/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/envs/medgs-gh200/bin/python /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/repo/MedGS/render.py --model_path /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/single_phase_ct_every_second_slice_v1__112_HM10395__phase_20__series_036286623964/medgs_img_poly2_30000_v1/model --iteration -1 --interp 2 --pipeline img --camera mirror --poly_degree 2 --chunks 1
Looking for config file in /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/single_phase_ct_every_second_slice_v1__112_HM10395__phase_20__series_036286623964/medgs_img_poly2_30000_v1/model/cfg_args
Config file found: /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/single_phase_ct_every_second_slice_v1__112_HM10395__phase_

Rendering progress: 100%|██████████| 26/26 [00:02<00:00, 12.89it/s]


Rendered PNG files: 52
Render directory:   /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/single_phase_ct_every_second_slice_v1__112_HM10395__phase_20__series_036286623964/medgs_img_poly2_30000_v1/model/render_img


## 9. Match rendered midpoints to held-out CT slices

With `--interp 2`, MedGS writes `_0` for the trained frame and `_1`
for the midpoint toward the next trained frame.

The `mirror` camera pipeline may render two camera views per frame.
This cell detects whether the renderer produced one or two views per
training frame and, when necessary, identifies the non-mirrored view.

In [12]:
print("Matching MedGS midpoint renders to held-out test slices.")


def load_png_gray(path: Path) -> np.ndarray:
    """Load a PNG as a normalized grayscale float array."""

    return np.asarray(
        Image.open(path).convert("L"),
        dtype=np.float32,
    ) / 255.0


base_render_files = sorted(RENDER_ROOT.glob("*_0.png"))
rendered_view_count = len(base_render_files)
training_frame_count = len(frame_manifest_df)

if rendered_view_count == training_frame_count:
    camera_stride = 1
    original_camera_offset = 0
elif rendered_view_count == 2 * training_frame_count:
    camera_stride = 2

    first_input = load_png_gray(
        Path(frame_manifest_df.iloc[0]["OriginalPNG"])
    )
    first_view = load_png_gray(RENDER_ROOT / "00000_0.png")
    second_view = load_png_gray(RENDER_ROOT / "00001_0.png")

    first_error = np.mean((first_view - first_input) ** 2)
    second_error = np.mean((second_view - first_input) ** 2)

    original_camera_offset = int(second_error < first_error)
else:
    raise RuntimeError(
        f"Unexpected rendered view count: {rendered_view_count}. "
        f"Training frames: {training_frame_count}."
    )

coordinate_to_frame = dict(
    zip(
        frame_manifest_df["SliceCoordinate"],
        frame_manifest_df["FrameIndex"],
    )
)

reconstruction_rows = []

for _, row in test_manifest_df.iterrows():
    lower_coordinate = float(row["LowerTrainCoordinate"])
    upper_coordinate = float(row["UpperTrainCoordinate"])
    target_coordinate = float(row["SliceCoordinate"])
    expected_midpoint = 0.5 * (
            lower_coordinate + upper_coordinate
    )

    if not np.isclose(target_coordinate, expected_midpoint):
        raise ValueError(
            f"Slice {row['SliceIndex']} is not the midpoint between "
            f"its neighbouring training slices."
        )

    lower_frame_index = int(
        coordinate_to_frame[lower_coordinate]
    )
    upper_frame_index = int(
        coordinate_to_frame[upper_coordinate]
    )

    render_view_index = (
            original_camera_offset
            + camera_stride * lower_frame_index
    )
    reconstruction_path = (
            RENDER_ROOT / f"{render_view_index:05d}_1.png"
    )

    reconstruction_rows.append(
        {
            "SliceIndex": int(row["SliceIndex"]),
            "SliceCoordinate": target_coordinate,
            "GroundTruthPath": row["Path"],
            "LowerTrainFrame": lower_frame_index,
            "UpperTrainFrame": upper_frame_index,
            "LowerTrainPath": train_manifest_df.loc[
                train_manifest_df["SliceCoordinate"]
                == lower_coordinate,
                "Path",
            ].iloc[0],
            "UpperTrainPath": train_manifest_df.loc[
                train_manifest_df["SliceCoordinate"]
                == upper_coordinate,
                "Path",
            ].iloc[0],
            "ReconstructionPath": str(reconstruction_path),
        }
    )

reconstruction_manifest_df = pd.DataFrame(
    reconstruction_rows
).sort_values("SliceIndex").reset_index(drop=True)

print(f"Rendered views per training frame: {camera_stride}")
print(f"Original camera offset:           {original_camera_offset}")
print(
    f"Matched test reconstructions:      "
    f"{len(reconstruction_manifest_df)}"
)

display(reconstruction_manifest_df)

Matching MedGS midpoint renders to held-out test slices.
Rendered views per training frame: 1
Original camera offset:           0
Matched test reconstructions:      24


,SliceIndex,SliceCoordinate,GroundTruthPath,LowerTrainFrame,UpperTrainFrame,LowerTrainPath,UpperTrainPath,ReconstructionPath
0,1,-70.5,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,0,1,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
1,3,-64.5,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,1,2,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
2,5,-58.5,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,2,3,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
3,7,-52.5,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,3,4,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
4,9,-46.5,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,4,5,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
5,11,-40.5,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,5,6,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
6,13,-34.5,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,6,7,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
7,15,-28.5,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,7,8,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/data/raw/dicom_by_series/112_HM10395/1.3.6.1.4.1.14519.5.2.1.68...,/net/storage/pr3/plgrid/plggtriplane/plg

## 10. Compute reconstruction metrics

Metrics are calculated against the original held-out DICOM slices
after applying the same HU window used for training.

A linear midpoint interpolation between the neighbouring training
slices is evaluated as a baseline. LPIPS is calculated when the
`lpips` package is installed in the active kernel; otherwise its
columns contain `NaN`.

In [20]:
print("Computing MedGS and linear-interpolation metrics on test slices.")

lpips_model = lpips.LPIPS(net="alex").cuda().eval()
print("LPIPS enabled.")


def lpips_distance(
        reference: np.ndarray,
        prediction: np.ndarray,
) -> float:
    """Compute LPIPS for two normalized grayscale images."""

    if lpips_model is None:
        return float("nan")

    reference_tensor = (
            torch.from_numpy(reference)
            .float()
            .unsqueeze(0)
            .unsqueeze(0)
            .repeat(1, 3, 1, 1)
            .cuda()
            * 2.0
            - 1.0
    )
    prediction_tensor = (
            torch.from_numpy(prediction)
            .float()
            .unsqueeze(0)
            .unsqueeze(0)
            .repeat(1, 3, 1, 1)
            .cuda()
            * 2.0
            - 1.0
    )

    with torch.no_grad():
        return float(
            lpips_model(
                reference_tensor,
                prediction_tensor,
            ).item()
        )


metric_rows = []

for _, row in reconstruction_manifest_df.iterrows():
    ground_truth = window_hu(
        read_hu_image(Path(row["GroundTruthPath"])),
        HU_WINDOW_LOW,
        HU_WINDOW_HIGH,
    )
    reconstruction = load_png_gray(
        Path(row["ReconstructionPath"])
    )

    lower_train = window_hu(
        read_hu_image(Path(row["LowerTrainPath"])),
        HU_WINDOW_LOW,
        HU_WINDOW_HIGH,
    )
    upper_train = window_hu(
        read_hu_image(Path(row["UpperTrainPath"])),
        HU_WINDOW_LOW,
        HU_WINDOW_HIGH,
    )
    linear_baseline = 0.5 * (
            lower_train + upper_train
    )

    metric_rows.append(
        {
            "SliceIndex": int(row["SliceIndex"]),
            "SliceCoordinate": float(row["SliceCoordinate"]),
            "MedGS_PSNR": peak_signal_noise_ratio(
                ground_truth,
                reconstruction,
                data_range=1.0,
            ),
            "MedGS_SSIM": structural_similarity(
                ground_truth,
                reconstruction,
                data_range=1.0,
            ),
            "MedGS_LPIPS": lpips_distance(
                ground_truth,
                reconstruction,
            ),
            "Linear_PSNR": peak_signal_noise_ratio(
                ground_truth,
                linear_baseline,
                data_range=1.0,
            ),
            "Linear_SSIM": structural_similarity(
                ground_truth,
                linear_baseline,
                data_range=1.0,
            ),
            "Linear_LPIPS": lpips_distance(
                ground_truth,
                linear_baseline,
            ),
            "ReconstructionPath": row["ReconstructionPath"],
        }
    )

metrics_df = pd.DataFrame(metric_rows)
metrics_df.to_csv(METRICS_PATH, index=False)

metrics_summary_df = pd.DataFrame(
    [
        {
            "Method": "MedGS",
            "PSNR": metrics_df["MedGS_PSNR"].mean(),
            "SSIM": metrics_df["MedGS_SSIM"].mean(),
            "LPIPS": metrics_df["MedGS_LPIPS"].mean(),
        },
        {
            "Method": "Linear interpolation",
            "PSNR": metrics_df["Linear_PSNR"].mean(),
            "SSIM": metrics_df["Linear_SSIM"].mean(),
            "LPIPS": metrics_df["Linear_LPIPS"].mean(),
        },
    ]
)

print(f"Metrics saved to: {METRICS_PATH}")
display(metrics_summary_df)
display(metrics_df)

Computing MedGS and linear-interpolation metrics on test slices.
Setting up [LPIPS] perceptual loss: trunk [alex], v[0.1], spatial [off]


/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/envs/medgs-gh200/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/envs/medgs-gh200/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=AlexNet_Weights.IMAGENET1K_V1`. You can also use `weights=AlexNet_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /net/home/plgrid/plgmozo/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|████████████████████████████████████████████████████████████████████████████████████████| 233M/233M [00:01<00:00, 160MB/s]


Loading model from: /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/envs/medgs-gh200/lib/python3.11/site-packages/lpips/weights/v0.1/alex.pth
LPIPS enabled.
Metrics saved to: /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/single_phase_ct_every_second_slice_v1__112_HM10395__phase_20__series_036286623964/medgs_img_poly2_30000_v1/metrics.csv


,Method,PSNR,SSIM,LPIPS
0,MedGS,24.702241,0.653324,0.149296
1,Linear interpolation,25.039237,0.742586,0.111304


,SliceIndex,SliceCoordinate,MedGS_PSNR,MedGS_SSIM,MedGS_LPIPS,Linear_PSNR,Linear_SSIM,Linear_LPIPS,ReconstructionPath
0,1,-70.5,24.559226,0.631877,0.160103,25.190987,0.755874,0.105917,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
1,3,-64.5,24.166023,0.618576,0.158548,24.847615,0.750987,0.107977,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
2,5,-58.5,24.767122,0.630929,0.159512,25.452897,0.755863,0.102208,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
3,7,-52.5,24.760037,0.644559,0.149533,25.317998,0.758136,0.106585,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
4,9,-46.5,24.702369,0.651968,0.148737,25.326628,0.765007,0.106036,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
5,11,-40.5,25.113303,0.662293,0.145182,25.685270,0.770309,0.104015,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
6,13,-34.5,24.924395,0.662579,0.143545,25.469705,0.768498,0.104195,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
7,15,-28.5,24.617577,0.660613,0.142895,25.042736,0.765811,0.099956,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
8,17,-22.5,24.771239,0.656780,0.137912,25.391563,0.767372,0.095614,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...
9,19,-16.5,24.651468,0.658888,0.145825,25.080904,0.763067,0.107906,/net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/singl...


## 11. Interactive train/reconstruction sequence

Training positions display the actual HU-windowed DICOM input.
Test positions display the corresponding MedGS reconstruction.

In [29]:
print("Creating a fixed side-by-side train/reconstruction slice browser.")

from IPython.display import clear_output, display

reconstruction_by_slice = {
    int(row["SliceIndex"]): row
    for _, row in metrics_df.iterrows()
}


def show_slice_side_by_side(slice_position: int) -> None:
    """Display one fixed-size horizontal comparison canvas."""

    row = split_manifest_df.iloc[slice_position]

    slice_index = int(row["SliceIndex"])
    coordinate = float(row["SliceCoordinate"])

    if row["Split"] == "train":
        left_image = window_hu(
            read_hu_image(Path(row["Path"])),
            HU_WINDOW_LOW,
            HU_WINDOW_HIGH,
        )

        right_image = np.ones_like(left_image)

        left_label = "Train input"
        right_label = ""

        title_line_1 = f"Slice {slice_index} | TRAIN INPUT"
        title_line_2 = (
            f"z={coordinate:.3f} mm | "
            f"HU [{HU_WINDOW_LOW:g}, {HU_WINDOW_HIGH:g}]"
        )

    else:
        metric_row = reconstruction_by_slice[slice_index]

        left_image = load_png_gray(
            Path(metric_row["ReconstructionPath"])
        )

        right_image = window_hu(
            read_hu_image(Path(row["Path"])),
            HU_WINDOW_LOW,
            HU_WINDOW_HIGH,
        )

        left_label = "Reconstructed"
        right_label = "Original test slice"

        title_line_1 = f"Slice {slice_index} | RECONSTRUCTION"
        title_line_2 = (
            f"z={coordinate:.3f} mm | "
            f"PSNR={metric_row['MedGS_PSNR']:.2f} | "
            f"SSIM={metric_row['MedGS_SSIM']:.4f} | "
            f"LPIPS={metric_row['MedGS_LPIPS']:.4f}"
        )

    separator_width = 16

    separator = np.ones(
        (left_image.shape[0], separator_width),
        dtype=np.float32,
    )

    comparison_image = np.concatenate(
        [
            left_image,
            separator,
            right_image,
        ],
        axis=1,
    )

    figure, axis = plt.subplots(figsize=(12, 6.4))

    axis.imshow(
        comparison_image,
        cmap="gray",
        vmin=0.0,
        vmax=1.0,
    )

    axis.axis("off")

    axis.text(
        0.25,
        1.02,
        left_label,
        transform=axis.transAxes,
        ha="center",
        va="bottom",
        fontsize=14,
    )

    axis.text(
        0.75,
        1.02,
        right_label,
        transform=axis.transAxes,
        ha="center",
        va="bottom",
        fontsize=14,
    )

    figure.suptitle(
        f"{title_line_1}\n{title_line_2}",
        fontsize=16,
        y=0.98,
    )

    figure.subplots_adjust(
        left=0.01,
        right=0.99,
        bottom=0.01,
        top=0.82,
    )

    plt.show()
    plt.close(figure)


side_by_side_slider = widgets.IntSlider(
    value=0,
    min=0,
    max=len(split_manifest_df) - 1,
    step=1,
    description="Slice",
    continuous_update=False,
)

side_by_side_output = widgets.Output()


def update_side_by_side_browser(change=None) -> None:
    """Refresh the fixed comparison canvas."""

    with side_by_side_output:
        clear_output(wait=True)
        show_slice_side_by_side(side_by_side_slider.value)


side_by_side_slider.observe(
    update_side_by_side_browser,
    names="value",
)

display(side_by_side_slider, side_by_side_output)
update_side_by_side_browser()

Creating a fixed side-by-side train/reconstruction slice browser.


IntSlider(value=0, continuous_update=False, description='Slice', max=49)

Output()

## 12. Save the concise run record

In [30]:
print("Saving the experiment settings and aggregate results.")

run_record = {
    "schema_version": 1,
    "config_name": CONFIG_NAME,
    "case_name": SELECTED_CASE_NAME,
    "run_name": RUN_NAME,
    "case_config": str(
        selected_case_directory / "case.json"
    ),
    "split_manifest": str(
        selected_case_directory
        / loaded_case_config["manifest_file"]
    ),
    "dataset_root": str(MEDGS_DATASET_ROOT),
    "model_root": str(MODEL_ROOT),
    "render_root": str(RENDER_ROOT),
    "metrics_file": str(METRICS_PATH),
    "training": {
        "iterations": ITERATIONS,
        "poly_degree": POLY_DEGREE,
        "batch_size": BATCH_SIZE,
        "camera": CAMERA,
        "command": train_command,
    },
    "rendering": {
        "interpolation": RENDER_INTERPOLATION,
        "camera_stride": camera_stride,
        "original_camera_offset": original_camera_offset,
        "command": render_command,
    },
    "metrics_mean": {
        row["Method"]: {
            "PSNR": (
                None if pd.isna(row["PSNR"])
                else float(row["PSNR"])
            ),
            "SSIM": (
                None if pd.isna(row["SSIM"])
                else float(row["SSIM"])
            ),
            "LPIPS": (
                None if pd.isna(row["LPIPS"])
                else float(row["LPIPS"])
            ),
        }
        for _, row in metrics_summary_df.iterrows()
    },
}

RUN_CONFIG_PATH.write_text(
    json.dumps(run_record, indent=2) + "\n",
    encoding="utf-8",
)

print(f"Run record: {RUN_CONFIG_PATH}")
print(f"Metrics:    {METRICS_PATH}")
print(f"Model:      {MODEL_ROOT}")

Saving the experiment settings and aggregate results.
Run record: /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/single_phase_ct_every_second_slice_v1__112_HM10395__phase_20__series_036286623964/medgs_img_poly2_30000_v1/run.json
Metrics:    /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/single_phase_ct_every_second_slice_v1__112_HM10395__phase_20__series_036286623964/medgs_img_poly2_30000_v1/metrics.csv
Model:      /net/storage/pr3/plgrid/plggtriplane/plgmozo/medgs4d/results/experiments/single_phase_ct_every_second_slice_v1/single_phase_ct_every_second_slice_v1__112_HM10395__phase_20__series_036286623964/medgs_img_poly2_30000_v1/model
